# A2-Lite — why did feature teaching hurt Pedestrian?
Latest revision: **A2-LITE-FEATURE-ENDPOINT-GRADIENTS-2026-10-15-r1**.

This is a short, read-only diagnostic, NOT another training experiment. Compare the original initialization, the completed label-only epoch5, and the completed feature-KD epoch5 on the SAME32 TRAIN frames used by the original gradient calibration (16 Pedestrian-containing +16 Vehicle-only). Only32 images/labels/calibrations and their existing teacher features are staged. No full-dataset copy, teacher forward pass, native CUDA build, validation inference or optimizer updates.

Measure the actual weighted feature gradient versus label-learning gradients separately for Pedestrian and Vehicle, on the visual encoder and trainable late backbone. The original all-object matching, normalization, dimension compensation and per-frame teacher class shares are preserved. A class's classification contribution includes negative examples in that logit channel; the unused third channel is reported separately, not silently discarded.

Use the SAME fully ramped feature coefficient at all three states to isolate state changes. The initialization measurement is not a replay of its first warmup update. A 5% gradient calibration at initialization does not impose a 5% bound later. Negative cosines indicate locally opposing directions, not proof of the cause of validation AP regression. Gradients are measured before clipping/AdamW.

Choose a GPU (A100 preferred), open this fresh notebook and run sections1–6. Keep the completed feature run read-only. The results ZIP does NOT contain weights or teacher features: those must still be present in the original Drive run. A new diagnostic may use the current Colab GPU/Torch without restoring an old CUDA stack; its own runtime is recorded and frozen. On a later reset, rerun all sections to restore only32 local inputs. If this diagnostic's runtime changes, choose a new diagnostic RUN_ID rather than mixing frame reports.

## 1. Mount Drive, set paths and define every helper


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from collections import deque
from datetime import datetime, timezone
import json, re, shlex, subprocess, sys, zipfile
REVISION='A2-LITE-FEATURE-ENDPOINT-GRADIENTS-2026-10-15-r1'
RUN_ID='a2_lite_feature_endpoint_gradients_r1'
if not re.fullmatch(r'[a-z0-9_]+',RUN_ID):raise ValueError('Use lowercase letters, digits and underscores')
MOBILE_REPO=Path('/content/mobile_adas3d')
COMPLETED_RUN=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite_features/a2_lite_512_foreground_feature_kd_r1')
FEATURE_RUN=COMPLETED_RUN  # Or the exact a2_lite_feature_results.zip; weights/features remain separate.
FEATURE_FILES=COMPLETED_RUN/'teacher_features'
INITIALIZATION=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite/a2_lite_e1_d1_cpu_screen_r1/a2_lite_initialization.pth')
CONTROL_CHECKPOINT=COMPLETED_RUN/'control/last.pth'
KD_CHECKPOINT=COMPLETED_RUN/'kd/last.pth'
SOURCE=Path('/content/MonoDETR_A2LITE_FEATURE_AUDIT_'+RUN_ID)
DATASET=Path('/content/kitti_a2_lite_accuracy_feature_audit_'+RUN_ID)
WEIGHTS=Path('/content/a2_lite_feature_audit_weights_'+RUN_ID)
PREVIOUS_DATA=Path('/content/kitti_a2_lite_accuracy_feature_a2_lite_512_foreground_feature_kd_r1')
CANDIDATES=[PREVIOUS_DATA,Path('/content/kitti_a2_lite_accuracy_cache_prepare_v1'),Path('/content/kitti'),Path('/content/drive/MyDrive/datasets/kitti')]
FEATURE_CANDIDATES=[PREVIOUS_DATA/'teacher_feature_cache']
OUTPUT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/diagnostics/a2_lite_feature_gradients')/RUN_ID
if OUTPUT.resolve().is_relative_to(COMPLETED_RUN.resolve()):raise RuntimeError('Use a separate diagnostic output')
PYTHON=Path(sys.executable);SCRIPT=MOBILE_REPO/'scripts/audit_a2_lite_feature_gradients.py'
LOGS=OUTPUT/'colab_logs';LOGS.mkdir(parents=True,exist_ok=True)
READY=False;PREPARED=False;PROBED=False
def run_logged(command,name,cwd=None):
    command=list(map(str,command));path=LOGS/f"{name}_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')}.log"
    print('+',shlex.join(command),'\nDurable log:',path,flush=True);tail=deque(maxlen=40)
    with path.open('w') as log:
        log.write(shlex.join(command)+'\n');log.flush()
        process=subprocess.Popen(command,cwd=str(cwd) if cwd else None,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='',flush=True);log.write(line);log.flush();tail.append(line.rstrip())
        except BaseException:
            process.terminate()
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.kill();process.wait()
            raise
        finally:process.stdout.close()
        code=process.wait();log.write(f'\nEXIT CODE: {code}\n');log.flush()
    if code:raise RuntimeError(f'Exit {code}; full log={path}\n'+'\n'.join(tail))
    return path
def stage(name,*extra):
    if not READY and name!='bundle':raise RuntimeError('Run sections1 and2 first')
    return run_logged([PYTHON,'-X','faulthandler','-u',SCRIPT,name,'--output',OUTPUT,'--device','cuda',*extra],name,MOBILE_REPO)
print(REVISION,'\n32 TRAIN frames x3 states; ZERO optimizer updates.\nOutput:',OUTPUT)


## 2. Update main and check the current GPU
Install preprocessing dependencies, not Torch/CUDA or a historical virtual environment. Current diagnostic runtime may differ from training, but it must remain consistent across all three state probes. Missing checkpoints are caught before staging.


In [ ]:
READY=False
if not MOBILE_REPO.exists():run_logged(['git','clone','--branch','main','https://github.com/Ali-RT/mobile_adas3d.git',MOBILE_REPO],'clone')
dirty=subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=MOBILE_REPO,text=True).strip()
if dirty:raise RuntimeError('Preserve tracked edits and use clean main.\n'+dirty)
branch=subprocess.check_output(['git','branch','--show-current'],cwd=MOBILE_REPO,text=True).strip()
if branch!='main':raise RuntimeError(f'Use main, not {branch!r}')
run_logged(['git','pull','--ff-only'],'update',MOBILE_REPO)
if not SCRIPT.is_file():raise FileNotFoundError('Open the latest published feature-endpoint audit from main')
for path in (FEATURE_RUN,INITIALIZATION,CONTROL_CHECKPOINT,KD_CHECKPOINT):
    if not path.exists():raise FileNotFoundError(f'{path}: correct section1 paths. Results ZIPs contain no weights.')
if not any(p.is_dir() for p in [FEATURE_FILES,*FEATURE_CANDIDATES]):raise FileNotFoundError('Need the saved teacher_features folder from the completed run; do not regenerate features')
packages=['numpy==2.1.3','timm==1.0.27','onnxruntime==1.30.0','pillow==11.3.0','scipy==1.16.3','opencv-python-headless==5.0.0.93','pyyaml','scikit-image']
run_logged([PYTHON,'-m','pip','install','-q',*packages],'ordinary_dependencies')
run_logged([PYTHON,'-c',"import torch;assert torch.cuda.is_available(),'Select a GPU runtime';torch.cuda.init();print(torch.__version__,torch.cuda.get_device_name(0),'portable gradient audit; no native extension')"],'gpu_probe')
READY=True


## 3. Tests and efficient32-frame staging
Pin the exact reviewed run and both epoch5 checkpoints. Restore a private source checkout before using it; never require the completed run's deleted /content checkout. Copy only32 cached teacher feature files and96 image/label/calibration files with four I/O workers. Verified local files are reused after an interruption. No full7481-frame staging or teacher model loading.


In [ ]:
PREPARED=False
if not READY:raise RuntimeError('Run sections1 and2 first')
run_logged([PYTHON,'-m','unittest','discover','-s','tests','-p','test_a2_lite_feature_gradients.py','-v'],'audit_tests',MOBILE_REPO)
extra=[]
for candidate in CANDIDATES:extra += ['--candidate',candidate]
for candidate in FEATURE_CANDIDATES:extra += ['--feature-candidate',candidate]
stage('prepare','--feature-run',FEATURE_RUN,'--features-dir',FEATURE_FILES,'--initialization',INITIALIZATION,
      '--control-checkpoint',CONTROL_CHECKPOINT,'--kd-checkpoint',KD_CHECKPOINT,'--repo',SOURCE,
      '--dataset',DATASET,'--weights-dir',WEIGHTS,*extra)
MANIFEST=json.loads((OUTPUT/'feature_gradient_manifest.json').read_text())
assert MANIFEST['staged_train_frames']==32 and MANIFEST['staged_validation_frames']==0
assert MANIFEST['training_authorized'] is False
PREPARED=True
print('Current audit runtime:',MANIFEST['runtime'])
print('Original training runtime:',MANIFEST['historical_runtime'])
print('Frozen fully ramped feature coefficient:',MANIFEST['final_feature_coefficient'])


## 4. Measure three unchanged-weight states
Each state runs in a separate subprocess. autograd.grad measures directions without filling parameter .grad fields; there is no optimizer. State hashes cover weights and buffers before/after. Completed per-frame reports resume only with matching audit identity.


In [ ]:
PROBED=False
if not PREPARED:raise RuntimeError('Run sections1–3 first')
for role in ('initial','control_epoch5','feature_epoch5'):
    stage('probe','--role',role)
    report=json.loads((OUTPUT/f'{role}_gradient_report.json').read_text())
    assert report['optimizer_steps']==0 and report['weights_and_buffers_unchanged'] and report['frames']==32
    print(role,'—32 TRAIN probes, unchanged weights/buffers, zero optimizer updates')
    for row in report['summary']:
        if row['parameter_group']=='visual_encoder' and row['frame_stratum']=='all' and row['comparison'].startswith('kd_all'):
            print(json.dumps(row))
PROBED=True


## 5. Collect evidence — no new training decision
Norm ratio compares weighted teacher gradient size with label-learning gradient size on the same coordinates. Cosine below0 means locally opposing directions. Compare both sizes and directions across the three states and classes. Neither a large scalar feature loss nor one negative cosine establishes why AP changed. This report does not authorize a coefficient change, longer run or deployment.


In [ ]:
if not PROBED:raise RuntimeError('Run sections1–4 first')
stage('compare')
REPORT=json.loads((OUTPUT/'feature_gradient_comparison.json').read_text())
assert REPORT['complete'] and REPORT['optimizer_steps']==0 and REPORT['training_authorized'] is False
print(REPORT['next_step'])
print('No validation inference or new training was performed.')


## 6. Download evidence, including after a failure
Return **a2_lite_feature_gradient_results.zip**. Reports contain every frame's class-gradient comparison, state hashes, reviewed-run evidence, runtime identity and logs. Images, teacher tensors and checkpoints stay on disk and are excluded. If any earlier section fails, run this section to return the combined logs.


In [ ]:
BUNDLE=OUTPUT/'a2_lite_feature_gradient_results.zip'
try:
    if not SCRIPT.is_file():raise RuntimeError('Audit helper missing; package readable logs directly')
    stage('bundle')
except Exception as error:
    print('Bundle helper unavailable:',error,'— packaging readable logs/reports directly.')
    with zipfile.ZipFile(BUNDLE,'w',zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(OUTPUT.rglob('*')):
            if path.is_file() and path.suffix in ('.json','.txt','.log'):archive.write(path,path.relative_to(OUTPUT))
print(BUNDLE)
from google.colab import files
files.download(str(BUNDLE))
